## Model Wrapper

In [ ]:
Reusable AI Model Service Using Hugging Face
Objective: Convert notebook-only inference code into a reusable Python component using a ModelService class.
We will implement a sentiment classification service that supports:
- Model loading and configuration
- Input validation
- Text preprocessing
- Model inference
- Output processing
- Error handling
- Logging
The same class structure can later be reused for text classification, question answering, and other Hugging Face tasks.
1. Install libraries

In [ ]:
2. Implement the reusable ModelService class
Create a file named model_service.py, or run this code in a notebook cell.

In [1]:
import logging
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# -----------------------------------
# 1. Configure logging
# -----------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)


# -----------------------------------
# 2. Model configuration
# -----------------------------------

@dataclass
class ModelConfig:
    model_name: str = (
        "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
    )
    max_length: int = 128
    batch_size: int = 8
    device: str = "auto"


# -----------------------------------
# 3. Reusable model service
# -----------------------------------

class ModelService:

    def __init__(self, config: ModelConfig = None):
        self.config = config or ModelConfig()

        self.tokenizer = None
        self.model = None
        self.device = self._select_device()

        logger.info("ModelService initialized")

    # Select CPU or GPU
    def _select_device(self):
        if self.config.device == "auto":
            return torch.device(
                "cuda" if torch.cuda.is_available() else "cpu"
            )

        if self.config.device == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but unavailable")

        if self.config.device not in ("cpu", "cuda"):
            raise ValueError("Device must be auto, cpu, or cuda")

        return torch.device(self.config.device)

    # Load tokenizer and model
    def load_model(self):
        try:
            logger.info(
                "Loading model: %s", self.config.model_name
            )

            self.tokenizer = AutoTokenizer.from_pretrained(
                self.config.model_name
            )

            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.config.model_name
            )

            self.model.to(self.device)
            self.model.eval()

            logger.info("Model loaded successfully on %s", self.device)

        except Exception:
            logger.exception("Model loading failed")
            raise

    # Validate inputs
    def validate_input(self, texts):
        if isinstance(texts, str):
            texts = [texts]

        if not isinstance(texts, list):
            raise TypeError("Input must be a string or list of strings")

        if not texts:
            raise ValueError("Input list cannot be empty")

        if len(texts) > 1000:
            raise ValueError("Maximum 1000 texts per request")

        for text in texts:
            if not isinstance(text, str):
                raise TypeError("Every input must be a string")

            if not text.strip():
                raise ValueError("Input text cannot be empty")

        return texts

    # Preprocess text
    def preprocess(self, texts):
        logger.info("Preprocessing %d text inputs", len(texts))

        return self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.config.max_length,
            return_tensors="pt"
        ).to(self.device)

    # Run inference and process outputs
    def predict(self, texts):
        try:
            # Ensure model is loaded
            if self.model is None or self.tokenizer is None:
                self.load_model()

            # Input validation
            texts = self.validate_input(texts)

            results = []

            # Process inputs in batches
            for start in range(0, len(texts), self.config.batch_size):
                batch = texts[start:start + self.config.batch_size]

                # Preprocessing
                inputs = self.preprocess(batch)

                # Inference
                with torch.inference_mode():
                    outputs = self.model(**inputs)
                    probabilities = torch.softmax(
                        outputs.logits, dim=-1
                    )

                # Output processing
                for probs in probabilities:
                    class_id = int(torch.argmax(probs).item())
                    label = self.model.config.id2label[class_id]
                    score = float(probs[class_id].item())

                    results.append({
                        "label": label,
                        "score": round(score, 4)
                    })

            logger.info("Prediction completed for %d inputs", len(texts))
            return results

        except (TypeError, ValueError):
            logger.warning("Invalid prediction input", exc_info=True)
            raise

        except Exception:
            logger.exception("Inference failed")
            raise

In [ ]:
3. Use the reusable service
You can now use the class without rewriting the model-loading and inference code each time.

In [2]:
# Configure the model
config = ModelConfig(
    model_name="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    max_length=128,
    batch_size=4,
    device="auto"
)

# Create service
service = ModelService(config)

# Input data
texts = [
    "I love learning Python.",
    "The application is very slow.",
    "The experience was excellent.",
    "I am disappointed with the service."
]

# Generate predictions
results = service.predict(texts)

# Display results
for text, result in zip(texts, results):
    print("Text:", text)
    print("Prediction:", result["label"])
    print("Confidence:", result["score"])
    print("-" * 40)

2026-10-14 10:39:28,285 - INFO - ModelService initialized
2026-10-14 10:39:28,287 - INFO - Loading model: distilbert/distilbert-base-uncased-finetuned-sst-2-english
2026-10-14 10:39:28,735 - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english "HTTP/1.1 200 OK"
2026-10-14 10:39:29,017 - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-14 10:39:29,250 - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-14 10:39:29,540 - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

2026-10-14 10:39:32,551 - INFO - Model loaded successfully on cpu
2026-10-14 10:39:32,553 - INFO - Preprocessing 4 text inputs
2026-10-14 10:39:32,885 - INFO - Prediction completed for 4 inputs


Text: I love learning Python.
Prediction: POSITIVE
Confidence: 0.9994
----------------------------------------
Text: The application is very slow.
Prediction: NEGATIVE
Confidence: 0.9998
----------------------------------------
Text: The experience was excellent.
Prediction: POSITIVE
Confidence: 0.9999
----------------------------------------
Text: I am disappointed with the service.
Prediction: NEGATIVE
Confidence: 0.9998
----------------------------------------


In [ ]:
4. Test input validation and error handling

In [3]:
# Test 1: Empty input
try:
    service.predict("")
except ValueError as e:
    print("Validation error:", e)


# Test 2: Invalid input type
try:
    service.predict(123)
except TypeError as e:
    print("Validation error:", e)


# Test 3: Empty list
try:
    service.predict([])
except ValueError as e:
    print("Validation error:", e)

2026-10-14 10:39:32,914 - WARNING - Invalid prediction input
Traceback (most recent call last):
  File "C:\Users\new\AppData\Local\Temp\ipykernel_9668\2420166083.py", line 135, in predict
    texts = self.validate_input(texts)
  File "C:\Users\new\AppData\Local\Temp\ipykernel_9668\2420166083.py", line 111, in validate_input
    raise ValueError("Input text cannot be empty")
ValueError: Input text cannot be empty
2026-10-14 10:39:32,921 - WARNING - Invalid prediction input
Traceback (most recent call last):
  File "C:\Users\new\AppData\Local\Temp\ipykernel_9668\2420166083.py", line 135, in predict
    texts = self.validate_input(texts)
  File "C:\Users\new\AppData\Local\Temp\ipykernel_9668\2420166083.py", line 98, in validate_input
    raise TypeError("Input must be a string or list of strings")
TypeError: Input must be a string or list of strings
2026-10-14 10:39:32,927 - WARNING - Invalid prediction input
Traceback (most recent call last):
  File "C:\Users\new\AppData\Local\Temp\ipyke

Validation error: Input text cannot be empty
Validation error: Input must be a string or list of strings
Validation error: Input list cannot be empty


In [ ]:

Create reusable Python classes around model inference.
For example:

class ModelService:
    
    ...
The implementation should demonstrate:
Model loading
Configuration
Input validation
Preprocessing
Inference
Output processing
Error handling
Logging
The objective is to move from:
Notebook-only code
to:
Reusable AI component


In [ ]:
Component	Responsibility
ModelConfig	Stores model name, batch size, device and token length
load_model()	Loads the tokenizer and pretrained model
validate_input()	Checks incoming data
preprocess()	Converts text to model-ready tensors
predict()	Runs inference and returns structured results
Logging and exceptions	Helps diagnose failures